# Data Pre-processing of NSW Electricity Demand

## Section 1 - Objective

The purpose of this notebook is to prepare the NSW electricity demand dataset for forecasting models.

This includes:
1. converting the datetime field into a proper time index,
2. checking missing values,
3. confirming alignment across datasets,
4. selecting modelling variables,
5. splitting the data into training and testing sets.

## Section 2 - Import Libraries and Load Raw Data

In [124]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

fatal: destination path 'capstone_project_GroupA' already exists and is not an empty directory.


In [125]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

repo_path = "capstone_project_GroupA"
nsw_path = os.path.join(repo_path, "data", "NSW")

part_a = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partaa")
part_b = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partab")
forecast_zip = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip")

with open(forecast_zip, "wb") as outfile:
    for p in [part_a, part_b]:
        with open(p, "rb") as infile:
            outfile.write(infile.read())

df_demand = pd.read_csv(os.path.join(nsw_path, "totaldemand_nsw.csv.zip"))
df_temp = pd.read_csv(os.path.join(nsw_path, "temperature_nsw.csv.zip"))
df_forecast = pd.read_csv(os.path.join(nsw_path, "forecastdemand_nsw.csv.zip"))

print("Demand shape:", df_demand.shape)
print("Temp shape:", df_temp.shape)
print("Forecast shape:", df_forecast.shape)

Demand shape: (196513, 3)
Temp shape: (220326, 3)
Forecast shape: (10906019, 6)


## Section 4. Convert Datetime Format

Convert "LASTCHANGED" in table df_forecast to datetime and round it so it can align with the demand timestamps.

In [126]:
df_demand["DATETIME"] = pd.to_datetime(df_demand["DATETIME"], dayfirst=True)
df_temp["DATETIME"] = pd.to_datetime(df_temp["DATETIME"], dayfirst=True)
df_forecast["LASTCHANGED"] = pd.to_datetime(df_forecast["LASTCHANGED"])
df_forecast["DATETIME_ROUND"] = df_forecast["LASTCHANGED"].dt.round("30min")
print(df_demand.dtypes)
print(df_temp.dtypes)
print(df_forecast.dtypes)
df_forecast[["LASTCHANGED","DATETIME_ROUND"]].head()

DATETIME       datetime64[ns]
TOTALDEMAND           float64
REGIONID               object
dtype: object
LOCATION               object
DATETIME       datetime64[ns]
TEMPERATURE           float64
dtype: object
PREDISPATCHSEQNO             int64
REGIONID                    object
PERIODID                     int64
FORECASTDEMAND             float64
LASTCHANGED         datetime64[ns]
DATETIME                    object
DATETIME_ROUND      datetime64[ns]
dtype: object


,LASTCHANGED,DATETIME_ROUND
0,2009-12-30 12:31:49,2009-12-30 12:30:00
1,2009-12-30 13:01:43,2009-12-30 13:00:00
2,2009-12-30 13:31:36,2009-12-30 13:30:00
3,2009-12-30 14:01:44,2009-12-30 14:00:00
4,2009-12-30 14:31:35,2009-12-30 14:30:00


In [127]:
print(df_forecast.head())

   PREDISPATCHSEQNO REGIONID  PERIODID  FORECASTDEMAND         LASTCHANGED  \
0        2009123018     NSW1        71         7832.04 2009-12-30 12:31:49   
1        2009123019     NSW1        70         7832.04 2009-12-30 13:01:43   
2        2009123020     NSW1        69         7832.03 2009-12-30 13:31:36   
3        2009123021     NSW1        68         7832.03 2009-12-30 14:01:44   
4        2009123022     NSW1        67         7830.96 2009-12-30 14:31:35   

              DATETIME      DATETIME_ROUND  
0  2010-01-01 00:00:00 2009-12-30 12:30:00  
1  2010-01-01 00:00:00 2009-12-30 13:00:00  
2  2010-01-01 00:00:00 2009-12-30 13:30:00  
3  2010-01-01 00:00:00 2009-12-30 14:00:00  
4  2010-01-01 00:00:00 2009-12-30 14:30:00  


## Section 6. Aggregate Forecast Data by 30-Minute Interval

There may be multiple forecast rows within the same rounded timestamp, so aggregate them using mean.

In [128]:
df_forecast_30min = (
    df_forecast
    .groupby("DATETIME_ROUND")["FORECASTDEMAND"]
    .mean()
    .reset_index()
)
df_forecast_30min = df_forecast_30min.rename(
    columns={"DATETIME_ROUND":"DATETIME"}
)
df_temp_30min = df_temp.groupby("DATETIME")["TEMPERATURE"].mean().reset_index()

## Section 7. Merge the Three Datasets

Merge using DATETIME as the key.

In [129]:
df_merged = (
    df_demand
    .merge(df_temp_30min, on="DATETIME", how="inner")
    .merge(df_forecast_30min, on="DATETIME", how="inner")
)
df_merged.head()

,DATETIME,TOTALDEMAND,REGIONID,TEMPERATURE,FORECASTDEMAND
0,2010-01-01 00:00:00,8038.00,NSW1,23.1,7544.100179
1,2010-01-01 00:30:00,7809.31,NSW1,22.9,7433.778364
2,2010-01-01 01:00:00,7483.69,NSW1,22.6,7427.407037
3,2010-01-01 01:30:00,7117.23,NSW1,22.5,7433.393774
4,2010-01-01 02:00:00,6812.03,NSW1,22.5,7446.699231


## Section 8. Sort and Set Time Index

Sorting and setting time index is necessary because time-series models require an ordered time index.

In [130]:
df_merged = df_merged.sort_values("DATETIME")
df_merged = df_merged.set_index("DATETIME")
df_merged.head()

,TOTALDEMAND,REGIONID,TEMPERATURE,FORECASTDEMAND
DATETIME,,,,
2010-01-01 00:00:00,8038.00,NSW1,23.1,7544.100179
2010-01-01 00:30:00,7809.31,NSW1,22.9,7433.778364
2010-01-01 01:00:00,7483.69,NSW1,22.6,7427.407037
2010-01-01 01:30:00,7117.23,NSW1,22.5,7433.393774
2010-01-01 02:00:00,6812.03,NSW1,22.5,7446.699231


## Section 9. Check Missing Values

In [131]:
df_merged.isnull().sum()

,0
TOTALDEMAND,0
REGIONID,0
TEMPERATURE,0
FORECASTDEMAND,0


The merged dataset was checked for missing values using df_merged.isnull().sum().
No null values were found across all variables, indicating that the merging and alignment of the datasets were successful.

## Section 10. Remove Unnecessary Columns



In [132]:
df_merged["REGIONID"].unique()

array(['NSW1'], dtype=object)

The REGIONID variable contained only a single value (NSW1) across the entire dataset and therefore did not provide any predictive information. The column was removed from the dataset prior to model training.

In [133]:
df_merged = df_merged.drop(columns=["REGIONID"])

## Section 11. Reindex to Complete 30-Minute Timeline

The time index was examined to ensure continuity at the 30-minute frequency.
A total of 592 timestamps were missing from the expected sequence.
The dataset was therefore reindexed to a complete 30-minute timeline to ensure correct alignment when generating lag features.

In [134]:
# Section: Check missing time points

# expected full 30-minute timeline
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# find missing timestamps
missing_timepoints = full_index.difference(df_merged.index)

print("Expected number of time points:", len(full_index))
print("Actual number of time points:", len(df_merged.index))
print("Number of missing time points:", len(missing_timepoints))

# preview first few missing timestamps
print(missing_timepoints[:20])

Expected number of time points: 196512
Actual number of time points: 195920
Number of missing time points: 592
DatetimeIndex(['2010-01-10 04:00:00', '2010-01-11 17:00:00',
               '2010-01-14 13:30:00', '2010-01-15 10:30:00',
               '2010-01-16 10:30:00', '2010-01-19 00:30:00',
               '2010-01-19 10:30:00', '2010-01-20 15:30:00',
               '2010-01-21 18:00:00', '2010-01-23 08:30:00',
               '2010-01-23 18:30:00', '2010-01-24 21:30:00',
               '2010-02-01 19:00:00', '2010-02-03 12:00:00',
               '2010-02-05 03:00:00', '2010-02-05 12:00:00',
               '2010-02-06 20:00:00', '2010-02-09 10:00:00',
               '2010-02-10 14:30:00', '2010-02-10 20:00:00'],
              dtype='datetime64[ns]', freq=None)


In [135]:
time_gaps = df_merged.index.to_series().diff().value_counts().sort_index()
print(time_gaps)

DATETIME
0 days 00:30:00    195700
0 days 01:00:00       187
0 days 01:30:00        11
0 days 02:00:00         2
0 days 02:30:00         2
0 days 03:00:00         5
0 days 03:30:00         1
0 days 04:00:00         2
0 days 05:00:00         1
0 days 05:30:00         2
0 days 07:30:00         1
0 days 10:00:00         1
0 days 11:30:00         1
0 days 13:30:00         1
0 days 17:30:00         1
3 days 18:30:00         1
Name: count, dtype: int64


In [136]:
# Create full 30-minute time index
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# Reindex to full timeline
df_merged = df_merged.reindex(full_index)
df_merged.index.name = "DATETIME"

# Check missing values after reindexing
print(df_merged.isnull().sum())

TOTALDEMAND       592
TEMPERATURE       592
FORECASTDEMAND    592
dtype: int64


Missing numeric values in TOTALDEMAND, TEMPERATURE, and FORECASTDEMAND were filled using time-based interpolation. This method estimates missing observations using surrounding values while preserving the temporal structure of the data.

In [137]:
numeric_cols = ["TOTALDEMAND", "TEMPERATURE", "FORECASTDEMAND"]

df_merged[numeric_cols] = df_merged[numeric_cols].interpolate(method="time")

print(df_merged[numeric_cols].isnull().sum())

TOTALDEMAND       0
TEMPERATURE       0
FORECASTDEMAND    0
dtype: int64


## Section 13. Add Historical Demand Fields

Because the data is every 30 minutes:

1 day ago = 24 hrs x 2 = 48 rows before

1 week ago = 48 x 7 = 336 rows before

1 year ago = 48 x 365 = 17,520 rows before

In [138]:
df_merged["demand_1_day_ago"] = df_merged["TOTALDEMAND"].shift(48)
df_merged["demand_1_week_ago"] = df_merged["TOTALDEMAND"].shift(336)
df_merged["demand_1_year_ago"] = df_merged["TOTALDEMAND"].shift(17520)
df_merged[[
    "TOTALDEMAND",
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
]].head(1000000)

,TOTALDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,
2010-01-01 00:00:00,8038.00,NaN,NaN,NaN
2010-01-01 00:30:00,7809.31,NaN,NaN,NaN
2010-01-01 01:00:00,7483.69,NaN,NaN,NaN
2010-01-01 01:30:00,7117.23,NaN,NaN,NaN
2010-01-01 02:00:00,6812.03,NaN,NaN,NaN
...,...,...,...,...
2021-03-17 21:30:00,7503.12,7462.84,7732.69,7535.95
2021-03-17 22:00:00,7419.77,7373.83,7642.24,7480.90
2021-03-17 22:30:00,7417.91,7345.78,7567.36,7465.50


Rows with NaN entries in demand_1_day_ago, demand_1_week_ago, and demand_1_year_ago are removed to ensure that all lagged demand features are available for model training. These missing values occur at the beginning of the dataset where sufficient historical observations do not exist to compute the lag features.

In [139]:
df_model = df_merged.dropna(subset=[
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
])
df_model[[
    "TOTALDEMAND",
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
]].head(1000000)

,TOTALDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,
2011-01-01 00:00:00,8063.36,7628.57,7358.22,8038.00
2011-01-01 00:30:00,7844.62,7384.33,7141.33,7809.31
2011-01-01 01:00:00,7545.29,7161.66,6870.62,7483.69
2011-01-01 01:30:00,7206.58,6842.79,6557.80,7117.23
2011-01-01 02:00:00,6824.75,6594.82,6237.94,6812.03
...,...,...,...,...
2021-03-17 21:30:00,7503.12,7462.84,7732.69,7535.95
2021-03-17 22:00:00,7419.77,7373.83,7642.24,7480.90
2021-03-17 22:30:00,7417.91,7345.78,7567.36,7465.50


## Section 14. Train / Validation / Test Split

For time-series forecasting, use chronological split.

In [140]:
n = len(df_model)

train_size = int(n * 0.6)
val_size = int(n * 0.2)

train       = df_model.iloc[:train_size]
validation  = df_model.iloc[train_size:train_size + val_size]
test        = df_model.iloc[train_size + val_size:]

## Section 15. Feature Scaling

In [141]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

features = ["TOTALDEMAND",
            "demand_1_day_ago",
            "demand_1_week_ago",
            "demand_1_year_ago",
            "TEMPERATURE", "FORECASTDEMAND"]

train_scaled = scaler.fit_transform(train[features])
val_scaled = scaler.transform(validation[features])
test_scaled = scaler.transform(test[features])


Feature scaling is not required for statistical models such as SARIMAX, as these models operate directly on the original scale of the data. However, scaling may be applied when training neural network-based models such as PatchTST to improve training stability and convergence.

## Section 16. Saving Preprocessed Datasets for Modelling

After completing the preprocessing and feature engineering steps, the final datasets are exported as CSV files for reuse in the modelling stage. This step ensures that the preprocessing pipeline does not need to be recomputed repeatedly when training different models. The processed dataset (df_model) and its corresponding train, validation, and test splits are saved separately. In addition, scaled versions of the datasets are also stored for use in deep learning models such as LSTM and PatchTST, which require normalized input features for stable training.

Saving these datasets improves workflow reproducibility and modularity. Subsequent modelling notebooks can directly load the processed data without rerunning the entire preprocessing procedure. This approach also reduces computational overhead and helps maintain consistency across different model experiments.

The following datasets are saved:

- df_model.csv: Final dataset after preprocessing and feature engineering

- train.csv: Training dataset

- validation.csv: Validation dataset

- test.csv: Test dataset

- train_scaled.csv: Scaled training dataset for neural network models

- val_scaled.csv: Scaled validation dataset

- test_scaled.csv: Scaled test dataset

These files are stored in the directory:

capstone_project_GroupA/data/processed/

This structure allows different modelling notebooks (train.csv, test.csv and validation.csv for SARIMAX and train_scaled, test_scaled and val_scaled for LSTM, PatchTST) to load the same consistent datasets.

In [142]:
# Convert scaled numpy arrays into DataFrames so they can be saved as CSV files
# This preserves column names and datetime index for later modelling

features = [
    "TOTALDEMAND",
    "TEMPERATURE",
    "FORECASTDEMAND",
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
]

train_scaled = pd.DataFrame(train_scaled, columns=features, index=train.index)
val_scaled = pd.DataFrame(val_scaled, columns=features, index=validation.index)
test_scaled = pd.DataFrame(test_scaled, columns=features, index=test.index)


# ---------------------------------------------------------
# Save all processed datasets for use in modelling notebooks
# ---------------------------------------------------------

import os

repo_path = "capstone_project_GroupA"
output_path = os.path.join(repo_path, "data", "processed")

# Create folder if it does not exist
os.makedirs(output_path, exist_ok=True)

# Save main modelling dataset
df_model.to_csv(os.path.join(output_path, "df_model.csv"))

# Save train / validation / test splits
train.to_csv(os.path.join(output_path, "train.csv"))
validation.to_csv(os.path.join(output_path, "validation.csv"))
test.to_csv(os.path.join(output_path, "test.csv"))

# Save scaled datasets (used for LSTM and PatchTST models)
train_scaled.to_csv(os.path.join(output_path, "train_scaled.csv"))
val_scaled.to_csv(os.path.join(output_path, "val_scaled.csv"))
test_scaled.to_csv(os.path.join(output_path, "test_scaled.csv"))

# Display confirmation
print("Saved files to:", output_path)
print(os.listdir(output_path))

Saved files to: capstone_project_GroupA/data/processed
['test.csv', 'validation.csv', 'df_model.csv', 'train_scaled.csv', 'train.csv', 'test_scaled.csv', 'val_scaled.csv']


## Section 17. Version Control and Synchronization Using Git

To ensure that all generated datasets are accessible across different notebooks and computational environments, the processed data files are committed and synchronized with the project’s Git repository. Version control is managed using Git, which allows the project files to be tracked, updated, and shared across different development sessions.

The following Git commands are used to update the repository:

git add .

git commit -m "Add preprocessed data"

git push

By committing the processed datasets to the repository, the modelling notebooks can reliably load the required files without needing to recompute the preprocessing steps. This workflow also supports reproducibility, allowing instructors or collaborators to replicate the modelling process using the same prepared datasets.

In [147]:
%cd capstone_project_GroupA

/content/capstone_project_GroupA


In [148]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/NSW/forecastdemand_nsw.csv.zip
	data/processed/

nothing added to commit but untracked files present (use "git add" to track)


In [145]:
!git commit -m "Add preprocessed data"

fatal: not a git repository (or any of the parent directories): .git


In [149]:
!git add data/processed/*.csv

In [150]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/processed/df_model.csv
	new file:   data/processed/test.csv
	new file:   data/processed/test_scaled.csv
	new file:   data/processed/train.csv
	new file:   data/processed/train_scaled.csv
	new file:   data/processed/val_scaled.csv
	new file:   data/processed/validation.csv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/NSW/forecastdemand_nsw.csv.zip



In [151]:
!git commit -m "Add preprocessed datasets"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@1070b0f0f73d.(none)')
